# 04 — LangChain, testing, and the API flow

**Goal:** understand what LangChain composes and how fake dependencies protect the workflow.

```text
input dictionary → chat prompt → selected model → output parser → text
```

## Keep responsibilities separate

- **System message:** fixed behavior and grounding rules.
- **Human message:** changing context and question.
- **Model:** provider-specific execution behind one interface.
- **Parser:** converts the model message into string-compatible text.

LangChain is a composition toolbox. It does not replace the Harshu AI OS router, Chroma, FastAPI, or business rules.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer using only the supplied context. "
            "If evidence is missing, say you do not have enough information.",
        ),
        ("human", "Context:\n{context}\n\nQuestion:\n{question}"),
    ]
)

fake_model = FakeListChatModel(
    responses=["Chroma stores and searches embeddings."]
)
chain = prompt | fake_model | StrOutputParser()

result = chain.invoke(
    {
        "context": "Chroma stores and searches embeddings.",
        "question": "What does Chroma do?",
    }
)

print(result)
print(isinstance(result, str))

## Why fake models matter

A unit test should prove application behavior without depending on internet access, API keys, rate limits, or provider cost.

Use the Arrange–Act–Assert pattern:

1. **Arrange:** create fake dependencies and expected data.
2. **Act:** call one function.
3. **Assert:** check the returned behavior.

`monkeypatch` temporarily replaces a dependency during a pytest test and restores it afterward.

In [ ]:
expected_answer = "Chroma stores and searches embeddings."

# Arrange happened above when fake_model was created.
actual_answer = chain.invoke(
    {
        "context": expected_answer,
        "question": "What does Chroma do?",
    }
)

# Assert the contract, not the provider implementation.
assert isinstance(actual_answer, str)
assert actual_answer == expected_answer
print("test passed")

## Complete project flow

### Direct endpoint

```text
Frontend → POST /ask → classify → choose route → LiteLLM client → provider → answer
```

### RAG endpoint

```text
Frontend → POST /ask/rag → classify → choose route
         → embed question → Chroma top chunks → context + citations
         → LangChain prompt | selected model | parser
         → answer + retrieval evidence → frontend
```

A failure can come from classification, embeddings, retrieval, prompt/model execution, or response validation. Inspect the failing stage before changing code.

## Debugging map

| Symptom | First evidence to inspect |
|---|---|
| wrong model selected | classification and route dictionary |
| irrelevant context | retrieved IDs, distances, and source metadata |
| good context but bad answer | formatted prompt and model response |
| provider timeout | request ID, timeout, retry count, provider error |
| API validation error | Pydantic request/response shape |
| frontend network message | backend availability and browser network response |

## YOUR TURN

1. Change the fake response and prove the assertion fails before fixing it.
2. Print the formatted system and human messages separately.
3. Explain why `/ask` can stay direct while `/ask/rag` benefits from LangChain.
4. Starting at the frontend, explain the RAG flow using arrows without looking above.